# Part3: Advanced Features
# Qiskit* -> OpenQASM3* -> QubiC
*Qiskit and OpenQASM are open-source SDK and intermediate language developed by IBM

[OpenQASM](https://openqasm.com/intro.html) is a programming language for describing quantum computing algorithms and programs using gates, measurements, and classical control flow. It is commonly used as an intermediate representation to connect different tools or layers in a quantum control stack. This notebook shows how to use OpenQASM to export Qiskit circuits to run on QubiC, as well as how to write your own OpenQASM programs for QubiC.

*Note:* neither Qiskit (!) nor QubiC currently implements the full OpenQASM3 standard, but most commonly used features are supported.

Prerequisites: the QubiC software stack, `openpulse` `qiskit`, and `qiskit-qasm3-import`. Optionally, `matplotlib` with `pylatexenc` for drawing circuits.

In [ ]:
# standard Python modules, plus Qiskit with OpenQASM3 support
import io
import pprint

import qiskit as qk
import qiskit.qasm3 as qasm3

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# QubiC software
import distproc.compiler as qbcm
import qubitconfig.qchip as qbqc
import qubic.rfsoc.hwconfig as qbhwc
import qubic.rpc_client as qbr
import qubic.toolchain as qbtc

## Basic quantum circuit design using Qiskit

Qiskit is an open-source SDK for working with quantum computers at the level of extended quantum circuits, operators, and primitives, developed by IBM.

Detail documentation is located here: https://docs.quantum.ibm.com/

For complete tutorial checkout: https://github.com/Qiskit/qiskit-tutorials.git

### Quantum Logic gates

![logic.png](./images/Quantum_Logic_Gates.png)

Source: https://en.wikipedia.org/wiki/Quantum_logic_gate

In [ ]:
# Create a Quantum Circuit acting on a quantum register of three qubits and measurement
circ = qk.QuantumCircuit(2, 2)

In [ ]:
# Add a H gate on qubit $q_{0}$, putting this qubit in superposition.
circ.h(0)
# Add a CX (CNOT) gate on control qubit $q_{0}$ and target qubit $q_{1}$, putting
# the qubits in a Bell state.
circ.cx(0, 1)

In [ ]:
#Add a barrier before measurement
circ.barrier()

In [ ]:
# Measure all qubits
circ.measure_all()

In [ ]:
#Visualize the circuit
circ.draw('mpl')

## Exercise 1

In [ ]:
# Create a circuit consisting: 
# q0 - Hardmard gate
# q0, q1 - CX gate
# q1, q2 - CX gate
# Add a barrier
# Measure all qubits
#Visualize the circuit

## Quantum Teleportation
Suppose there are two researchers named Alice and Bob, and Alice wants to send her quantum state to Bob.

![teleport.svg](./images/teleport_circuit_full_gates.svg)


1. State preparation: Alice initializes her qubit to the state she wishes to teleport.

2. Shared entanglement: A Bell state is created and distributed to Alice and Bob (one qubit each).

3. Change of basis: Alice converts her two qubits from the Bell basis to the computational basis.

4. Measurement: Alice measures her two qubits, then tells Bob how to convert his qubit to obtain the desired state. Note that it is only quantum information being teleported, and not a physical particle.


Reference: https://pennylane.ai/qml/demos/tutorial_teleportation

### Circuit
This teleportation circuit shows the use of classical registers to capture measurement results and apply conditional logic on them. You can either use individual qubits (`QuantumRegister`s of size 1) or arrays of qubits (as done here: register of size 3) and the same goes for the classical bits to store the results. The result of measuring a qubit is stored in a classical bit, ie. 0 or 1; QASM3 does not currently support qudits. Internally, however, QubiC will run its own measurement protocol exactly as you have it configured, with the QASM variables only providing labeling for the control flow: they are not used to specify actual storage.

In [ ]:
# teleportation circuit example

q = qk.QuantumRegister(3) 
m = qk.ClassicalRegister(3)

#Circuit with 3 qubits
circuit = qk.QuantumCircuit(q, m)

#State preparation
circuit.u(0, 1, 2, q[0])

#Shared Entaglement between qubit 1 and 2
circuit.h(q[1])
circuit.cx(q[1], q[2])

#Change of basis
circuit.cx(q[0], q[1])
circuit.h(q[0])

for i in [1,0]:
    circuit.measure(q[i], m[i])

with circuit.if_test((m[1], 1)):
    circuit.x(q[2])
with circuit.if_test((m[0], 1)):
    circuit.z(q[2])

circuit.measure(q[2], m[2])

In [ ]:
# display the circuit; note that the classical register as drawn is indexed
circuit.draw(output='mpl')

## Helper functions for Qiskit -> QubiC

In [ ]:
# configure QubiC; this assumes the existence of the channel configuration and calibration files (adjust as needed)
fpga_config = qbhwc.FPGAConfig()
channel_config = qbhwc.load_channel_configs('config/channel_config.json')
qchip = qbqc.QChip('config/qubitcfg.json')

In [ ]:
# helpers for converting OpenQASM3 programs to Qiskit circuits and vice versa
def qiskit2qasm(circuit: qk.QuantumCircuit):
    """Convert a Qiskit circuit to OpenQASM3"""
    buf = io.StringIO()
    qasm3.dump(circuit, buf)
    buf.seek(0)
    return buf.read()

def qasm2qiskit(opq: str):
    """Convert an OpenQASM3 program to a Qiskit circuit"""
    return qasm3.loads(oqp)

In [ ]:
# convert the Qiskit circuit to an OpenQASM3 program, with the helper defined above.
oqp_ex1 = qiskit2qasm(circuit)
print(oqp_ex1)

**Exercise**: Compare the OpenQASM program with the original circuit and match the statements in the program with those constructing the circuit.

In [ ]:
# load QubiC OpenQASM3 support
import distproc.openqasm as qbqasm

In [ ]:
# convert the generated OpenQASM program to a QubiC progpram
qubic_program = qbqasm.load_qasm(oqp_ex1, qchip)
pprint.pprint(qubic_program)

**Exercise**: Compare the generated QubiC program with the OpenQASM program and/or Qiskit circuit and match its elements with statements in the others.

## Qubit naming
The labeling of qubits in generated QASM files is a bit ambiguous. This is especially true if you generate multiple programs, as each will have their own set of auto-generated labels (meaning, two separate conversions of the same circuit may generate different labels), and/or mix array style declarations with individual qubit declarations. In the QASM conversion to QubiC, heuristics are used that assume the standard QubiC labeling in the configuration files where qubits are named `Qn` with `n` a number starting from `0`.

If you need more control, if for example you want to address specific qubits (eg. `Q3`, `Q5`, and `Q7`, say, rather than qubits 0-2), you can either declare individual qubits in the Qiskit program and label them explicitly using their `name` argument; or you can supply a qubit mapping. (A third option is to change the channel configuration in QubiC, but doing so would be confusing, at least for enumerated qubits, and is thus not recommended.)

**Exercise**: Implement a custom qubit mapper that uses Q3, Q4, and Q5 instead of Q0-2.

In [ ]:
oqp_ex2 = oqp_ex1[:]    # re-use the QASM program from previous example

In [ ]:
# write a custom qubit mapping class
class MyQubitMap(qbqasm.QASMQubitMap):    # alternatively, use the QubitMap abstract base class
    def get_hardware_qubit(self, qubit_reg: str, index: int):
        # basic working example, offset all qubit indices by 3
        return super().get_hardware_qubit(qubit_reg, index+3)

qubic_program = qbqasm.load_qasm(oqp_ex2, qchip, qubit_map=MyQubitMap())
pprint.pprint(qubic_program)

## Custom gates
QASM standard gates are explicitly supported and will be decomposed as appropriate, e.g. RX and RY rotations are transformed into ZX90ZX90Z sequences. Additionally, all gates that exist in your QubiC calibration file are implicitly available for use in the QASM program: simply match up the gate name and the qubit that it is applied to with the labeling in the configuration file. This enables the use of any type of custom waveforms. Alternatively, you can supply a custom gate map.

**Exercise**: Implement a custom qubit mapper that uses Q3, Q4, and Q5 instead of Q0-2.

In [ ]:
oqp_ex3a = """
OPENQASM 3.0;
include "stdgates.inc";

bit out;

my_custom q[1];
out = measure q[1];
"""

In [ ]:
# write a custom gate mapping class
class MyGateMap(qbqasm.QASMGateMap):    # alternatively, use the GateMap abstract base class
    def get_qubic_gateinstr(self, gatename: str,
        hw_qubits: list, params: list=None, options: dict={}) -> list:

        # add a custom gate to your qubic configuration and decompose it here
        if gatename == "my_custom":
            # ... decompose as appropriate, then return the list of hardware gates;
            # as an example, the below turns "mycustom" into a U3 followed by a Hadamard
            return super().get_qubic_gateinstr("U", hw_qubits, [1, 2, 3], options) + \
                   super().get_qubic_gateinstr("h", hw_qubits, options=options)

        # all other gates
        return super().get_qubic_gateinstr(gatename, hw_qubits, params, options)

qubic_program = qbqasm.load_qasm(oqp_ex3a, qchip, gate_map=MyGateMap())
pprint.pprint(qubic_program)

Another alternative is to implement custom gates in QASM if they can be expressed in existing gates. This could be eg. a composite addressing multiple qubits, or it could be a generic or even parametrized gate expressed in `U3`s. For example:

In [ ]:
# custom gate example
oqp_ex3b = """
OPENQASM 3.0;
include "stdgates.inc";

gate my_composite a, b {
    h a;
    x b;
    cx a, b;
}
        
bit out;
qubit[2] q;

my_composite q[1], q[0];
out = measure q[1];
"""

qubic_program = qbqasm.load_qasm(oqp_ex3b, qchip)
pprint.pprint(qubic_program)

## Durations and delays
A big improvement in OpenQASM3 over OpenQASM2 is the introduction of programmable timing constraints. This part of the specification is very large and complex part and therefore seldom fully supported by tools, e.g. Qiskit does not support durations at this time. QubiC supports the basics and, if computations on the FPGA allow, duration-based control flow.

For example, durations can be used to create a T1 experiment, as shown below.

A couple of important implementation notes:
  - The measurements need not be captured individually in the QASM code since, as explained with Example 1, QubiC will store all measurement results for offline use. QASM still requires a placeholder however, `c` here, which will be elided by the QASM loader as it is unused.
  - This particular loop will be unrolled, because the delay is the result of a floating point computation, which the FPGA does not support. To make this possible, the `stride` and `ndata` variables must be declared const.
  - If all operations in the loop body can be handled on the FPGA (eg. only integer arithmatic or conditionals), the generated QubiC code will retain the loop.

In [ ]:
# T1 experiment
oqp_ex4 = """
OPENQASM 3.0;
include "stdgates.inc";

const duration stride = 100ns;
const int ndata = 10;

qubit[1] q;
bit c;

for int i in [0:ndata] {
    reset q[0];
    x q[0];
    delay[i * stride] q[0];
    c = measure q[0];
}"""

qubic_program = qbqasm.load_qasm(oqp_ex4, qchip)
pprint.pprint(qubic_program)